# Triton on SageMaker - NLP DeBERTa-v3

Deploy [microsoft/deberta-v3-base](https://huggingface.co/microsoft/deberta-v3-base) to NVIDIA Triton Inference Server on Amazon SageMaker using the ONNX Runtime backend.

DeBERTa-v3 improves on BERT with disentangled attention and an enhanced mask decoder. We export the model to ONNX for optimized inference, package it for Triton, and deploy it on a SageMaker endpoint.

This notebook was tested with the `conda_python3` kernel on an Amazon SageMaker notebook instance of type `g5`.

## Contents

1. [Set up the environment](#Set-up-the-environment)
1. [Export DeBERTa-v3 to ONNX](#Export-DeBERTa-v3-to-ONNX)
1. [Prepare the Triton model repository](#Prepare-the-Triton-model-repository)
1. [Package and upload to S3](#Package-and-upload-to-S3)
1. [Create SageMaker Endpoint](#Create-SageMaker-Endpoint)
1. [Run inference](#Run-inference)
1. [Terminate endpoint and clean up](#Terminate-endpoint-and-clean-up)

## Set up the environment

Install dependencies and configure the SageMaker session and IAM role.

In [ ]:
!pip install -U pip boto3 "sagemaker>=2,<3" transformers sentencepiece torch onnx onnxscript
!pip install "tritonclient[http]" numpy protobuf

In [ ]:
import boto3
import json
import numpy as np
import sagemaker
import time

from sagemaker import get_execution_role

sess = boto3.Session()
sm = sess.client("sagemaker")
sagemaker_session = sagemaker.Session(boto_session=sess)
client = boto3.client("sagemaker-runtime")
region = sess.region_name

role = get_execution_role()

print(f"Region: {region}")
print(f"Role:   {role}")

In [ ]:
# Triton DLC images use a single account ID across all regions.
# See: https://aws.github.io/deep-learning-containers/reference/available_images/
TRITON_ACCOUNT_ID = "763104351884"

base = "amazonaws.com.cn" if region.startswith("cn-") else "amazonaws.com"
triton_image_uri = (
    f"{TRITON_ACCOUNT_ID}.dkr.ecr.{region}.{base}"
    f"/sagemaker-tritonserver:24.09-py3"
)

print(f"Triton image URI: {triton_image_uri}")

## Export DeBERTa-v3 to ONNX

We use `torch.onnx.export` to export `microsoft/deberta-v3-base` to ONNX format, following the same approach as the BERT example in this lab.

The model is exported with a sequence classification head. Replace `MODEL_ID` with your own fine-tuned checkpoint as needed.

In [ ]:
import os
import torch
from transformers import DebertaV2ForSequenceClassification, DebertaV2Tokenizer

MODEL_ID = "microsoft/deberta-v3-base"
ONNX_OUTPUT_DIR = "workspace/deberta_onnx"
os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

model = DebertaV2ForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

NUM_LABELS = model.config.num_labels  # 2 for the base model

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_ID)

bs = 1
seq_len = 128
dummy_input_ids = torch.randint(1000, (bs, seq_len))
dummy_attention_mask = torch.ones(bs, seq_len, dtype=torch.int64)

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    os.path.join(ONNX_OUTPUT_DIR, "model.onnx"),
    export_params=True,
    opset_version=18,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence"},
        "attention_mask": {0: "batch_size", 1: "sequence"},
        "logits": {0: "batch_size"},
    },
)

tokenizer.save_pretrained(ONNX_OUTPUT_DIR)

print(f"ONNX model saved to {ONNX_OUTPUT_DIR}/")
print(f"NUM_LABELS: {NUM_LABELS}")

### Validate the ONNX model locally

Quick sanity check before deploying.

In [ ]:
import onnxruntime as ort
import numpy as np

ort_session = ort.InferenceSession(os.path.join(ONNX_OUTPUT_DIR, "model.onnx"))

text = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."
encoded = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="np")
inputs = {
    "input_ids": encoded["input_ids"].astype(np.int64),
    "attention_mask": encoded["attention_mask"].astype(np.int64),
}
logits = ort_session.run(["logits"], inputs)[0]

print(f"Input: {text}")
print(f"Logits shape: {logits.shape}")
print(f"Logits: {logits}")
print(f"\nNote: This is a base model (not fine-tuned), so predictions are not meaningful.")

## Prepare the Triton model repository

Triton expects a specific directory layout. We write a `config.pbtxt` that configures the ONNX Runtime backend with dynamic batching, then copy the exported model into the version directory.

```
deberta/
├── config.pbtxt
└── 1/
    ├── model.onnx
    └── model.onnx.data
```

**Note**: SageMaker expects the model tarball to have a top-level directory matching `SAGEMAKER_TRITON_DEFAULT_MODEL_NAME`. The `.data` file contains the model weights (PyTorch dynamo export stores them separately).

In [ ]:
import os
import shutil

MODEL_NAME = "deberta"
TRITON_MODEL_DIR = f"triton-serve-deberta/{MODEL_NAME}"

os.makedirs(f"{TRITON_MODEL_DIR}/1", exist_ok=True)

# Write config.pbtxt
config_pbtxt = f"""name: "{MODEL_NAME}"
backend: "onnxruntime"
max_batch_size: 32

input [
  {{
    name: "input_ids"
    data_type: TYPE_INT64
    dims: [-1]
  }},
  {{
    name: "attention_mask"
    data_type: TYPE_INT64
    dims: [-1]
  }}
]

output [
  {{
    name: "logits"
    data_type: TYPE_FP16
    dims: [{NUM_LABELS}]
  }}
]

instance_group {{
  count: 1
  kind: KIND_GPU
}}

dynamic_batching {{
  preferred_batch_size: [4, 8, 16]
  max_queue_delay_microseconds: 100
}}
"""

with open(f"{TRITON_MODEL_DIR}/config.pbtxt", "w") as f:
    f.write(config_pbtxt)

# Copy ONNX model and external weight data
shutil.copy2(f"{ONNX_OUTPUT_DIR}/model.onnx", f"{TRITON_MODEL_DIR}/1/model.onnx")

# PyTorch dynamo export stores weights in a separate .data file
onnx_data = f"{ONNX_OUTPUT_DIR}/model.onnx.data"
if os.path.exists(onnx_data):
    shutil.copy2(onnx_data, f"{TRITON_MODEL_DIR}/1/model.onnx.data")

# Print the directory structure
for root, dirs, files in os.walk(TRITON_MODEL_DIR):
    level = root.replace(TRITON_MODEL_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{indent}  {file} ({size_mb:.1f} MB)")

## Package and upload to S3

In [ ]:
!tar -C triton-serve-deberta/ -czf model.tar.gz deberta
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-deberta")
print(f"Model artifact uploaded to: {model_uri}")

## Create SageMaker Endpoint

Create a SageMaker model, endpoint configuration, and endpoint using the Triton container image.

The `SAGEMAKER_TRITON_DEFAULT_MODEL_NAME` environment variable tells Triton which model directory to load. Its value must match the top-level folder name in the model tarball.

In [ ]:
sm_model_name = "triton-deberta-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

container = {
    "Image": triton_image_uri,
    "ModelDataUrl": model_uri,
    "Environment": {"SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": MODEL_NAME},
}

create_model_response = sm.create_model(
    ModelName=sm_model_name, ExecutionRoleArn=role, PrimaryContainer=container
)

print("Model Arn: " + create_model_response["ModelArn"])

In [ ]:
endpoint_config_name = "triton-deberta-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "InstanceType": "ml.g5.2xlarge",
            "InitialVariantWeight": 1,
            "InitialInstanceCount": 1,
            "ModelName": sm_model_name,
            "VariantName": "AllTraffic",
        }
    ],
)

print("Endpoint Config Arn: " + create_endpoint_config_response["EndpointConfigArn"])

In [ ]:
endpoint_name = "triton-deberta-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)

print("Endpoint Arn: " + create_endpoint_response["EndpointArn"])

In [ ]:
resp = sm.describe_endpoint(EndpointName=endpoint_name)
status = resp["EndpointStatus"]
print("Status: " + status)

while status == "Creating":
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=endpoint_name)
    status = resp["EndpointStatus"]
    print("Status: " + status)

print("Arn: " + resp["EndpointArn"])
print("Status: " + status)

## Run inference

Once the endpoint is in service, we send requests using the [KFServing inference protocol](https://github.com/triton-inference-server/server/blob/main/docs/protocol/README.md) with the **binary+json** payload format for low-latency inference.

### Utility: Tokenize text and build payloads

In [ ]:
import tritonclient.http as httpclient
from transformers import DebertaV2Tokenizer

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_ID)
MAX_LENGTH = 128


def tokenize_text(text, max_length=MAX_LENGTH):
    """Tokenize text and return numpy arrays of input_ids and attention_mask."""
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="np",
    )
    return encoded["input_ids"].astype(np.int64), encoded["attention_mask"].astype(np.int64)


def build_binary_payload(input_ids, attention_mask):
    """Build a binary+json payload for Triton."""
    seq_len = input_ids.shape[1]
    inputs = [
        httpclient.InferInput("input_ids", [1, seq_len], "INT64"),
        httpclient.InferInput("attention_mask", [1, seq_len], "INT64"),
    ]
    inputs[0].set_data_from_numpy(input_ids, binary_data=True)
    inputs[1].set_data_from_numpy(attention_mask, binary_data=True)

    outputs = [httpclient.InferRequestedOutput("logits", binary_data=True)]

    request_body, header_length = httpclient.InferenceServerClient.generate_request_body(
        inputs, outputs=outputs
    )
    return request_body, header_length

### Inference with binary+json payload

The `binary+json` format sends tensor data as raw bytes instead of JSON-encoded arrays, which also supports FP16 output.
SageMaker uses a custom Content-Type header to indicate the JSON header size:
`application/vnd.sagemaker-triton.binary+json;json-header-size={N}`

In [ ]:
text = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."
input_ids, attention_mask = tokenize_text(text)
request_body, header_length = build_binary_payload(input_ids, attention_mask)

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/vnd.sagemaker-triton.binary+json;json-header-size={}".format(
        header_length
    ),
    Body=request_body,
)

# Parse the binary+json response
header_length_prefix = "application/vnd.sagemaker-triton.binary+json;json-header-size="
header_length_str = response["ContentType"][len(header_length_prefix):]

result = httpclient.InferenceServerClient.parse_response_body(
    response["Body"].read(), header_length=int(header_length_str)
)

logits = result.as_numpy("logits")
print(f"Input: {text}")
print(f"Logits: {logits}")

## Terminate endpoint and clean up

Delete the endpoint, endpoint configuration, and model to avoid ongoing charges.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=sm_model_name)
print("Cleanup complete.")

In [ ]:
# Optional: clean up local artifacts
!rm -rf triton-serve-deberta/deberta/1/model.onnx triton-serve-deberta/deberta/1/model.onnx.data
!rm -f model.tar.gz
print("Local artifacts cleaned up.")